# Geographical Aggregations for Pointwise events

In [ ]:
### Required Libraries 
import os
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from scipy import stats
import h3
import folium
from folium.plugins import HeatMap



pd.set_option("display.max_columns", None)

### Parameters 

# General 
INPUT_FILE = "../data/raw/carpetasFGJ.csv"
SELECTED_CRIME = "ROBO A TRANSEUNTE EN VÍA PÚBLICA CON Y SIN VIOLENCIA" # "ROBO A CASA HABITACIÓN CON VIOLENCIA"

PROC_DATA_DIR = "../data/proc/"

OUT_DIR = "../docs/resources/crime_eda"
PD_SHP_PATH = "../data/spatial/pd/09mun.shp"
STREETS_SHP_PATH = "../data/spatial/pd/09e.shp"
BUILDINGS_SHP_PATH = "../data/spatial/buildings/open_buildings_CVE_MUN_016.geojson"


os.makedirs(OUT_DIR, exist_ok=True)


SELECTED_COLUMNS = ["delito", "categoria_delito", "mes_inicio", "agencia",
                     "municipio_hecho", "anio_inicio", "mes_hecho", "anio_hecho",
                     "fecha_inicio", 
                     "hora_hecho", "fecha_hecho", "latitud", "longitud"]

START_YEAR = 2016
SELECTED_YEAR = 2019
SELECTED_MUN = "016"

BBOX = {
    "lat_lo":  19.04, "lat_hi":  19.75,
    "lon_lo": -99.40, "lon_hi": -98.95,
}

# Palette 
BLUE    = "#2166ac"
QUAL    = ["#2166ac","#4dac26","#d01c8b","#f1a340","#998ec3","#ca0020"]
SEQ     = LinearSegmentedColormap.from_list("seq", ["#f7fbff","#2166ac"])
GRAY    = "#b2b2b2"   
BG      = "#ffffff"
GRIDC   = "#ebebeb"

plt.rcParams.update({
    "figure.facecolor":   BG,
    "axes.facecolor":     BG,
    "axes.edgecolor":     GRIDC,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.spines.left":   False,
    "axes.spines.bottom": True,
    "axes.axisbelow":     True,
    "grid.color":         GRIDC,
    "grid.linewidth":     0.6,
    "text.color":         "#2d3436",
    "axes.labelcolor":    "#636363",
    "xtick.color":        "#636363",
    "ytick.color":        "#636363",
    "xtick.major.size":   0,
    "ytick.major.size":   0,
    "font.family":        "sans-serif",
    "font.size":          10,
    "axes.titlesize":     12,
    "axes.titleweight":   "bold",
    "axes.titlepad":      12,
    "legend.frameon":     False,
    "legend.fontsize":    9,
})


# DBSCAN model
EPSG_PROJ   = "EPSG:6372"   
EPSILON_M   = 500     # neighbourhood radius in metres
MIN_SAMPLES = 35      # min core-point density
ALPHA_HULL  = 0.001    # concavity parameter 





In [ ]:
import math
import geopandas as gpd
from shapely.geometry import LineString
from shapely.ops import substring

import math
import geopandas as gpd
from shapely.ops import substring

from shapely.geometry import Point, MultiPoint, LineString                                                                         
from shapely.ops import split                                                                                                      
                                                                                                                                    
                                                                                                                                    
def split_streets_at_intersections(streets_gdf, id_col="STREET_ID"):                                                               
    """                                                                                                                            
    Split street geometries at all intersections while preserving                                                                  
    the original street ID.                                                                                                        
                                                                                                                                    
    Returns a new GeoDataFrame with IDs:                                                                                           
        STREET_ID_0001, STREET_ID_0002, ...                                                                                        
    """                                                                                                                            
                                                                                                                                    
    streets = streets_gdf.copy()                                                                                                   
                                                                                                                                    
    # Convert MultiLineStrings to individual LineStrings                                                                           
    streets = streets.explode(index_parts=False).reset_index(drop=True)                                                            
                                                                                                                                    
    split_rows = []                                                                                                                
                                                                                                                                    
    # Spatial index for speed                                                                                                      
    sindex = streets.sindex                                                                                                        
                                                                                                                                    
    for idx, row in streets.iterrows():                                                                                            
        line = row.geometry                                                                                                        
                                                                                                                                    
        # Find nearby candidates                                                                                                   
        candidate_idxs = list(sindex.intersection(line.bounds))                                                                    
        candidate_idxs = [i for i in candidate_idxs if i != idx]                                                                   
                                                                                                                                    
        intersection_points = []                                                                                                   
                                                                                                                                    
        for cand_idx in candidate_idxs:                                                                                            
            other = streets.geometry.iloc[cand_idx]                                                                                
                                                                                                                                    
            inter = line.intersection(other)                                                                                       
                                                                                                                                    
            if inter.is_empty:                                                                                                     
                continue                                                                                                           
                                                                                                                                    
            if inter.geom_type == "Point":                                                                                         
                intersection_points.append(inter)                                                                                  
                                                                                                                                    
            elif inter.geom_type == "MultiPoint":                                                                                  
                intersection_points.extend(inter.geoms)                                                                            
                                                                                                                                    
            elif inter.geom_type == "GeometryCollection":                                                                          
                intersection_points.extend(                                                                                        
                    g for g in inter.geoms if g.geom_type == "Point"                                                               
                )                                                                                                                  
                                                                                                                                    
        # Remove duplicates                                                                                                        
        unique_pts = []                                                                                                            
        seen = set()                                                                                                               
                                                                                                                                    
        for pt in intersection_points:                                                                                             
            key = (round(pt.x, 8), round(pt.y, 8))                                                                                 
            if key not in seen:                                                                                                    
                seen.add(key)                                                                                                      
                unique_pts.append(pt)                                                                                              
                                                                                                                                    
        # Do not split at endpoints                                                                                                
        start = Point(line.coords[0])                                                                                              
        end = Point(line.coords[-1])                                                                                               
                                                                                                                                    
        split_pts = [                                                                                                              
            pt for pt in unique_pts                                                                                                
            if not pt.equals(start) and not pt.equals(end)                                                                         
        ]                                                                                                                          
                                                                                                                                    
        if split_pts:                                                                                                              
            try:                                                                                                                   
                parts = split(line, MultiPoint(split_pts)).geoms                                                                   
            except Exception:                                                                                                      
                parts = [line]                                                                                                     
        else:                                                                                                                      
            parts = [line]                                                                                                         
                                                                                                                                    
        for part_num, part in enumerate(parts, start=1):                                                                           
            new_row = row.copy()                                                                                                   
            new_row.geometry = part                                                                                                
            new_row[id_col] = f"{row[id_col]}_{part_num:04d}"                                                                      
            split_rows.append(new_row)                                                                                             
                                                                                                                                    
    return gpd.GeoDataFrame(split_rows, crs=streets.crs)


def split_gdf_by_length(gdf, max_length, id_col):
    """
    Split LineString geometries longer than max_length into multiple rows.

    Parameters
    ----------
    gdf : geopandas.GeoDataFrame
        Input GeoDataFrame.
    max_length : float
        Maximum segment length.
    id_col : str
        Column containing the identifier to modify.

    Returns
    -------
    geopandas.GeoDataFrame
    """
    rows = []

    for _, row in gdf.iterrows():
        geom = row.geometry
        base_id = row[id_col]

        if geom.length <= max_length:
            rows.append(row.copy())
            continue

        n_parts = math.ceil(geom.length / max_length)

        for i in range(n_parts):
            start_dist = i * max_length
            end_dist = min((i + 1) * max_length, geom.length)

            segment = substring(geom, start_dist, end_dist)

            new_row = row.copy()
            new_row.geometry = segment
            new_row[id_col] = f"{base_id}_{i + 1}"

            rows.append(new_row)

    return gpd.GeoDataFrame(rows, crs=gdf.crs)

In [ ]:
# Location gpd loads 

pd_gdf = gpd.read_file(PD_SHP_PATH).query(f""" CVE_MUN == '{SELECTED_MUN}'""")
CRS_AOI = pd_gdf.crs

df = pd.read_csv(INPUT_FILE)

# Basic Cleaning 
df["anio_inicio"] = pd.to_numeric(df["anio_inicio"], errors="coerce").astype("Int64")
df["anio_hecho"] = pd.to_numeric(df["anio_hecho"], errors="coerce").astype("Int64")



# Filter by crime of interes 

df_streets = (
    df
    .query(f""" categoria_delito == 'ROBO A TRANSEUNTE EN VÍA PÚBLICA CON Y SIN VIOLENCIA' and anio_hecho >= {START_YEAR}""")
    .filter(SELECTED_COLUMNS)
    .dropna()
)

df_builds = (
    df
    .query(f""" categoria_delito == 'ROBO A NEGOCIO CON VIOLENCIA' and anio_hecho >= {START_YEAR}""")
    .filter(SELECTED_COLUMNS)
    .dropna()
)

# Model basis to geopandas 

gdf_streets = (
    gpd.GeoDataFrame(                                                                                                            
        df_streets,                                                                                                                            
        geometry=gpd.points_from_xy(df_streets["longitud"], df_streets["latitud"]),                                                                    
        crs="EPSG:4326"  # WGS84 coordinate system                                                                                     
    )
    .query(f"anio_hecho == {SELECTED_YEAR}")
    .drop(columns=["anio_hecho"])
    .loc[
        lambda df: df.to_crs(CRS_AOI).within(pd_gdf.geometry.iloc[0])
    ]
    )

gdf_builds  = (
    gpd.GeoDataFrame(                                                                                                            
        df_builds,                                                                                                                            
        geometry=gpd.points_from_xy(df_builds["longitud"], df_builds["latitud"]),                                                                    
        crs="EPSG:4326"  # WGS84 coordinate system                                                                                     
    ) 
    .query(f"anio_hecho == {SELECTED_YEAR}")
    .drop(columns=["anio_hecho"])
    .loc[
        lambda df: df.to_crs(CRS_AOI).within(pd_gdf.geometry.iloc[0])
    ]
    )

In [ ]:
# Load the street level data 

STREETS_SHP_PATH = "../data/spatial/pd/09e.shp"      

STREET_LENGHT_CUT = 50
                                                                                                                                    
streets_gdf = (                                                                                                                    
    gpd.read_file(STREETS_SHP_PATH)   
    .query(f""" CVE_MUN == '{SELECTED_MUN}' """)                                                                                             
    .assign(STREET_ID = lambda x: x["CVEGEO"] + x["CVE_ENT"] + x["CVE_MUN"] + x["CVE_LOC"] + x["CVEVIAL"] + x["CVESEG"])           
    .drop(columns=["CVEGEO", "CVE_ENT", "CVE_MUN", "CVE_LOC", "CVEVIAL", "CVESEG"])                                                
    )                                                                                                                              
                                                                                                                                                                                                                                                              
# Corner Splits                                                                                                                                     
streets_split = split_streets_at_intersections(                                                                                    
    streets_gdf,                                                                                                                   
    id_col="STREET_ID"                                                                                                             
).rename(columns={"STREET_ID":"CVEGEO"})


# Crime DataBase 
street_crime_gdf = gpd.GeoDataFrame(    
    # Append Geometry
    streets_split[["CVEGEO","geometry"]]
    .merge(
        ## Original crime aggregation
        gdf_streets
        # Metters projection to INEGIs ESPG
        .to_crs(streets_split.crs)
        # Nearest join for avoid case specific geo mismatch 
        .sjoin_nearest(
            streets_split
            .filter(["CVEGEO","geometry"])
            ,how="left"
        )
        .drop(columns="geometry")
        # Group CVEGEO level 
        .groupby(["CVEGEO"]).agg(
            cnt=("fecha_inicio","count")
        )
        .reset_index()
        , on = "CVEGEO", how="left"
    )
    , crs=streets_split.crs, geometry="geometry"
)






In [ ]:
build_gdf = gpd.read_file(BUILDINGS_SHP_PATH)

positive_list = list(
    gdf_builds[["geometry"]]
    .sjoin_nearest(
            build_gdf
            ,how="left"
    )
    ["id"]
    .values
)

build_gdf["y"] = np.where(build_gdf.id.isin(positive_list), 1, 0)

In [ ]:
import folium
import branca.colormap as cm

street_crime_gdf = street_crime_gdf.to_crs(4326)
street_crime_gdf["cnt"] = street_crime_gdf["cnt"].fillna(0).astype(int)

colormap = cm.linear.Reds_05.scale(
    street_crime_gdf["cnt"].min(),
    street_crime_gdf["cnt"].max()
)

m = folium.Map(
    location=[
        street_crime_gdf.geometry.centroid.y.mean(),
        street_crime_gdf.geometry.centroid.x.mean()
    ],
    zoom_start=12,
    tiles="CartoDB positron"
)

# Buildings polygons
for _, row in build_gdf.to_crs(4326).iterrows():
    folium.Polygon(
        locations=[(y, x) for x, y in row.geometry.exterior.coords],
        fill_color="Blue",
        fill_opacity=0.2,
    ).add_to(m)

# Street crime GeoJson
folium.GeoJson(
    street_crime_gdf,
    style_function=lambda feature: {
        "color": colormap(feature["properties"]["cnt"]),
        "weight": 8,
        "opacity": 0.8,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["cnt"],
        aliases=["Count:"]
    ),
).add_to(m)

# Flagged buildings
folium.GeoJson(
    build_gdf[build_gdf["y"] == 1],
    style_function=lambda _: {
        "fillColor": "red",
        "color": "red",
        "weight": 2,
        "fillOpacity": 0.5,
    },
).add_to(m)

for _, row in gdf_streets.to_crs(4326).iterrows():
    folium.Circle(
        location=(row.geometry.y, row.geometry.x),
        fill_color="orange",
        color = "black",
        fill_opacity=0.5,
        radius=1,
    ).add_to(m)

colormap.caption = "Crime count"
colormap.add_to(m)
m